# ==============================================================================
# 🩺 CLINICAL SYMPTOM CHECKER: MULTI-DISEASE DIAGNOSIS VIA BERNOULLI NAIVE BAYES
# ==============================================================================
### Mathematical Formulation:
Bernoulli Naive Bayes is designed specifically for **binary presence/absence** feature indicators $x_i \in \{0, 1\}$:
$$P(X \mid C_k) = \prod_{i=1}^d p_{ki}^{x_i} (1 - p_{ki})^{(1 - x_i)}$$
Unlike Multinomial Naive Bayes (which counts frequencies), Bernoulli Naive Bayes **explicitly penalizes the non-occurrence** of an indicator feature via $(1 - p_{ki})^{(1 - x_i)}$.
If a patient does NOT exhibit a cardinal symptom (e.g., absence of high fever in acute malaria), Bernoulli NB actively down-weights that hypothesis.


In [1]:
# STEP 1: IMPORTS & SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Clinical Symptom Analytics Environment initialized.")


✅ STEP 1: Clinical Symptom Analytics Environment initialized.


## 📥 STEP 2 & 3: INGESTION & DATASET STRUCTURING
Loading the Kaggle Disease & Symptom Description dataset containing 41 pathological conditions.


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/disease_symptoms/dataset.csv'
df = pd.read_csv(data_path)

# Extract symptoms as a clean space-separated string per patient consultation
symptom_cols = [c for c in df.columns if c.startswith('Symptom')]
df['symptom_text'] = df[symptom_cols].apply(
    lambda row: ' '.join([str(val).strip().replace(' ', '_') for val in row.dropna()]),
    axis=1
)

print(f"📊 Dataset Shape: {df.shape[0]:,} patient cases")
print(f"📋 Unique Diseases: {df['Disease'].nunique()}")
df[['Disease', 'symptom_text']].head(3)


📊 Dataset Shape: 4,920 patient cases
📋 Unique Diseases: 41


,Disease,symptom_text
0,Fungal infection,itching skin_rash nodal_skin_eruptions dischro...
1,Fungal infection,skin_rash nodal_skin_eruptions dischromic__pat...
2,Fungal infection,itching nodal_skin_eruptions dischromic__patches


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating symptom text and disease diagnosis.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df['symptom_text']
y = df['Disease']

print(f"Null entries: {X.isnull().sum()}")
print(f"Total disease classes: {y.nunique()}")


Null entries: 0
Total disease classes: 41


## 🔒 STEP 6: STRATIFIED SPLIT
80/20 train/test split stratified across all 41 diagnostic conditions.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Cases: {len(X_train):,}")
print(f"Test Cases : {len(X_test):,}")


Train Cases: 3,936
Test Cases : 984


## ⚙️ STEP 7: BINARY SYMPTOM INDICATOR VECTORIZER
Using `CountVectorizer(binary=True)` to produce true binary symptom indicators $x_i \in \{0, 1\}$.


In [5]:
# STEP 7: BINARY VECTORIZER
vectorizer = CountVectorizer(binary=True, token_pattern=r'\S+')
print("✅ STEP 7: Binary Symptom Feature Indicator configured.")


✅ STEP 7: Binary Symptom Feature Indicator configured.


## ⚠️ STEP 8: THE ABSENCE PENALTY EXPERIMENT (Bernoulli vs Multinomial)
Demonstrating how BernoulliNB accounts for absent symptoms compared to MultinomialNB.


In [6]:
# STEP 8: BASELINE BERNOULLI FIT
pipe_bnb = Pipeline([
    ('vec', vectorizer),
    ('bnb', BernoulliNB(alpha=1.0))
])
pipe_bnb.fit(X_train, y_train)

acc_bnb = accuracy_score(y_test, pipe_bnb.predict(X_test))
print(f"BernoulliNB (alpha=1.0) Test Accuracy: {acc_bnb*100:.2f}%")


BernoulliNB (alpha=1.0) Test Accuracy: 100.00%


## 🎯 STEP 9: TUNING SMOOTHING PARAMETER $\alpha$
Optimizing Laplace smoothing with Stratified 5-Fold Cross Validation.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'bnb__alpha': np.logspace(-3, 1, 15)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    pipe_bnb,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Alpha Parameter: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Accuracy: {grid.best_score_*100:.2f}%")


🏆 Best Alpha Parameter: {'bnb__alpha': np.float64(0.001)}
🎯 Best 5-Fold CV Accuracy: 100.00%


## ⚔️ STEP 10: ALGORITHM DUEL (BernoulliNB vs Logistic Regression)
Comparing binary probabilistic diagnosis against multiclass Logistic Regression.


In [8]:
# STEP 10: MODEL BENCHMARK
pipe_lr = Pipeline([
    ('vec', vectorizer),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)

y_pred_bnb = best_model.predict(X_test)
y_pred_lr = pipe_lr.predict(X_test)

print("="*65)
print(f"BernoulliNB (Tuned)  -> Test Accuracy: {accuracy_score(y_test, y_pred_bnb)*100:.2f}% | Macro F1: {f1_score(y_test, y_pred_bnb, average='macro')*100:.2f}%")
print(f"Logistic Regression  -> Test Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}% | Macro F1: {f1_score(y_test, y_pred_lr, average='macro')*100:.2f}%")
print("="*65)


BernoulliNB (Tuned)  -> Test Accuracy: 100.00% | Macro F1: 100.00%
Logistic Regression  -> Test Accuracy: 100.00% | Macro F1: 100.00%


## 📊 STEP 11: DIAGNOSTIC ACCURACY & EVALUATION
Summary diagnostic evaluation across all diseases.


In [9]:
# STEP 11: EVALUATION REPORT
print(f"Overall Test Accuracy: {accuracy_score(y_test, y_pred_bnb)*100:.2f}%")
print(f"Macro Average F1-Score: {f1_score(y_test, y_pred_bnb, average='macro')*100:.2f}%")
print("\nTop 10 Disease Classification Excerpt:")
report = classification_report(y_test, y_pred_bnb, output_dict=True)
df_rep = pd.DataFrame(report).transpose().head(10)
print(df_rep[['precision', 'recall', 'f1-score']])


Overall Test Accuracy: 100.00%
Macro Average F1-Score: 100.00%

Top 10 Disease Classification Excerpt:
                                         precision  recall  f1-score
(vertigo) Paroymsal  Positional Vertigo        1.0     1.0       1.0
AIDS                                           1.0     1.0       1.0
Acne                                           1.0     1.0       1.0
Alcoholic hepatitis                            1.0     1.0       1.0
Allergy                                        1.0     1.0       1.0
Arthritis                                      1.0     1.0       1.0
Bronchial Asthma                               1.0     1.0       1.0
Cervical spondylosis                           1.0     1.0       1.0
Chicken pox                                    1.0     1.0       1.0
Chronic cholestasis                            1.0     1.0       1.0


## 💾 STEP 12 & 13: SERIALIZATION & CLINICAL INFERENCE SMOKE TEST
Deploying the symptom checker pipeline and testing triage latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/disease_symptom_bernoulli_nb_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_patient_symptoms = [
    "itching skin_rash nodal_skin_eruptions dischromic__patches"
]

t0 = time.perf_counter()
pred_disease = loaded_pipe.predict(sample_patient_symptoms)[0]
probs = loaded_pipe.predict_proba(sample_patient_symptoms)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🩺 LIVE CLINICAL SYMPTOM INFERENCE:")
print(f"   Diagnostic Hypothesis : 🏥 {pred_disease}")
print(f"   Diagnostic Confidence : {probs.max()*100:.2f}%")
print(f"   Triage Latency        : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/disease_symptom_bernoulli_nb_pipeline.joblib (96,841 bytes)

🩺 LIVE CLINICAL SYMPTOM INFERENCE:
   Diagnostic Hypothesis : 🏥 Fungal infection
   Diagnostic Confidence : 100.00%
   Triage Latency        : 1.767 ms (SLA < 2.0ms: PASS)
